# 6장. 데이터를 보며 질문을 만드는 EDA

이 노트북은 `notebooks/ch06_eda_questions.ipynb` 강의안을 실습하며 과제 답안을 작성한 자료다.

## 제출 정보

이름: 박성주 / GitHub ID: mikeypark / 작성일: 2026-10-16

## 학습 목표

5장에서 만든 전처리 데이터로 고객, 상품, 주문의 기본 분포를 먼저 보고, 매출처럼 돈이 걸린 지표는 꼭 `completed` 상태로 범위를 좁혀서 집계한 다음 그 합계가 어긋나지 않는지 끝에서 확인하는 것을 목표로 잡았다. 결과가 나올 때마다 그게 그냥 눈에 보이는 사실인지, 아니면 내가 거기에 이유를 붙인 건지 구분해서 적으려고 했다.

## 1. 패키지와 경로 설정

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'chapter06' else CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)

현재 실행 위치: C:\Users\uncle\Desktop\서울대 응용공학과\수업\2학기\대규모언어모델\git\llm-data-analysis-study\chapter06
프로젝트 루트: C:\Users\uncle\Desktop\서울대 응용공학과\수업\2학기\대규모언어모델\git\llm-data-analysis-study
전처리 데이터 폴더: C:\Users\uncle\Desktop\서울대 응용공학과\수업\2학기\대규모언어모델\git\llm-data-analysis-study\data\processed
보고서 폴더: C:\Users\uncle\Desktop\서울대 응용공학과\수업\2학기\대규모언어모델\git\llm-data-analysis-study\reports


## 2. 전처리된 데이터 불러오기

파일이 없으면 먼저 터미널에서 `python scripts/preprocess_data.py`를 실행해야 한다.

In [2]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [p for p in required_files if not p.exists()]
if missing_files:
    raise FileNotFoundError('전처리 파일이 없습니다: ' + ', '.join(str(p) for p in missing_files))

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
products = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('전처리 데이터 불러오기 완료')

전처리 데이터 불러오기 완료


## 3. 데이터 크기·컬럼·날짜 확인

In [3]:
data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

for name, df in data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()

orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
if 'signup_date' in customers.columns:
    customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')
if 'order_month' not in orders.columns:
    orders['order_month'] = orders['order_date'].dt.to_period('M').astype(str)

print('order_date 변환 실패:', orders['order_date'].isna().sum())
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())

[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (100, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 7)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']

[order_items]
shape: (764, 6)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']

order_date 변환 실패: 0
주문 시작일: 2025-07-09 00:00:00
주문 종료일: 2026-07-08 00:00:00


### 결과 관찰

5장에서 저장한 파일을 다시 불러왔는데도 customers (150, 6), products (100, 4), orders (300, 7), order_items (764, 6)로 행과 열 수가 그대로였다. orders와 order_items의 열이 늘어난 건 5장에서 만든 order_month, order_dayofweek, line_total이 그대로 남아 있기 때문이다. order_date 변환 실패는 0건이었고 주문 기간은 2025-07-09부터 2026-07-08까지였다.

CSV를 다시 읽으면 날짜 컬럼이 문자열로 돌아올 수 있다고 해서 `pd.to_datetime`을 한 번 더 걸었는데, 변환 실패가 안 나온 걸 보면 이번 파일에는 그 문제가 없었던 것 같다.

## 4. EDA 질문을 지표로 정리하기

표를 먼저 만들고 시작하면 분석이 산만해지지 않을 것 같아서, 고객·상품·주문·매출·시간·고객 가치 순으로 질문을 정리했다.

In [4]:
questions = pd.DataFrame({
    'analysis_area': ['고객', '상품', '주문', '매출', '시간', '고객 가치'],
    'question': [
        '고객은 어떤 지역과 연령대에 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '주문 상태와 결제수단 분포는 어떤가?',
        '카테고리별 completed 매출은 어떻게 다른가?',
        '월별 completed 매출과 주문 수는 어떻게 변하는가?',
        'completed 구매 금액이 높은 고객은 누구인가?',
    ],
    'analysis_scope': [
        'customers 전체',
        'products 전체',
        'orders 전체',
        "order_items, products (order_status == 'completed')",
        "orders, order_items (order_status == 'completed')",
        "customers, orders, order_items (order_status == 'completed')",
    ],
    'metric': [
        '고객 수, 평균 나이, 도시 비율',
        '카테고리별 상품 수, 평균 가격',
        '주문 상태/결제수단 건수, 비율',
        'completed 매출, 매출 비중',
        'completed 월별 매출, 월별 주문 수',
        'completed 고객별 총매출, 주문 횟수',
    ],
    'verification': [
        '도시별 합계 == 전체 고객 수',
        '카테고리별 합계 == 전체 상품 수',
        '상태별 합계 == 전체 주문 수',
        'completed 원본 line_total 합계 == 카테고리 합계',
        'completed 원본 line_total 합계 == 월별 합계',
        'completed 원본 line_total 합계 == 고객별 합계',
    ],
})

questions

,analysis_area,question,analysis_scope,metric,verification
0,고객,고객은 어떤 지역과 연령대에 분포하는가?,customers 전체,"고객 수, 평균 나이, 도시 비율",도시별 합계 == 전체 고객 수
1,상품,어떤 카테고리의 상품이 많은가?,products 전체,"카테고리별 상품 수, 평균 가격",카테고리별 합계 == 전체 상품 수
2,주문,주문 상태와 결제수단 분포는 어떤가?,orders 전체,"주문 상태/결제수단 건수, 비율",상태별 합계 == 전체 주문 수
3,매출,카테고리별 completed 매출은 어떻게 다른가?,"order_items, products (order_status == 'comple...","completed 매출, 매출 비중",completed 원본 line_total 합계 == 카테고리 합계
4,시간,월별 completed 매출과 주문 수는 어떻게 변하는가?,"orders, order_items (order_status == 'completed')","completed 월별 매출, 월별 주문 수",completed 원본 line_total 합계 == 월별 합계
5,고객 가치,completed 구매 금액이 높은 고객은 누구인가?,"customers, orders, order_items (order_status =...","completed 고객별 총매출, 주문 횟수",completed 원본 line_total 합계 == 고객별 합계


### 왜 이 질문을 선택했는가

여섯 개 질문을 고객·상품·주문처럼 테이블 하나로 끝나는 질문과, 매출·시간·고객 가치처럼 여러 테이블을 엮어야 답이 나오는 질문으로 나눠서 적었다. 뒤쪽 세 개는 전부 매출이 걸려 있어서 처음부터 completed 범위로 적어뒀는데, 주문 상태를 보니 취소·환불이 38% 가까이 차지하고 있어서 상태를 정하지 않고 집계하면 숫자가 부풀려질 걸 알았기 때문이다. 검증 방법 칸을 미리 채워둔 것도, 결과가 나왔을 때 집계가 맞게 됐는지 바로 확인할 수 있게 하려고 한 거다.

## 5. 고객 데이터 EDA

In [5]:
customers['age'].describe()

count    150.000000
mean      42.086667
std       15.613166
min       19.000000
25%       29.000000
50%       40.000000
75%       57.000000
max       69.000000
Name: age, dtype: float64

In [6]:
customer_city = customers['city'].value_counts(dropna=False).reset_index()
customer_city.columns = ['city', 'customer_count']
customer_city['customer_ratio'] = (
    customer_city['customer_count'] / customer_city['customer_count'].sum() * 100
).round(2)

customer_city

,city,customer_count,customer_ratio
0,성남,21,14.00
1,광주,17,11.33
2,부산,16,10.67
3,대구,15,10.00
4,서울,15,10.00
5,울산,14,9.33
6,인천,14,9.33
7,대전,14,9.33
8,수원,13,8.67
9,고양,11,7.33


In [7]:
customer_gender = customers['gender'].value_counts(dropna=False).reset_index()
customer_gender.columns = ['gender', 'customer_count']
customer_gender['customer_ratio'] = (
    customer_gender['customer_count'] / customer_gender['customer_count'].sum() * 100
).round(2)

customer_gender

,gender,customer_count,customer_ratio
0,F,84,56.0
1,M,66,44.0


### 결과 관찰

고객 150명의 나이는 19세부터 69세까지 퍼져 있고 평균은 42세, 중앙값은 40세였다. 도시는 성남이 21명(14%)으로 제일 많고 고양이 11명(7.33%)으로 제일 적었는데 차이가 크진 않았다. 성별은 여성이 84명(56%), 남성이 66명(44%)이었다.

도시별 고객 수 차이가 생각보다 크지 않아서, 이 데이터만 보고는 특정 지역에 쏠린 고객 기반이라고 말하기는 어려울 것 같다. 다만 고객 수가 많다고 그 도시 매출도 높을 거라는 보장은 없으니 이건 나중에 따로 확인해야 한다고 생각했다(-> 과제 1).

## 6. 상품 데이터 EDA

In [8]:
product_category = products['category'].value_counts(dropna=False).reset_index()
product_category.columns = ['category', 'product_count']
product_category['product_ratio'] = (
    product_category['product_count'] / product_category['product_count'].sum() * 100
).round(2)

product_category

,category,product_count,product_ratio
0,스포츠,19,19.0
1,전자기기,17,17.0
2,생활용품,16,16.0
3,뷰티,16,16.0
4,도서,14,14.0
5,패션,11,11.0
6,식품,7,7.0


In [9]:
products['price'].describe()

count       100.000000
mean     110040.000000
std       56433.910574
min        5000.000000
25%       65750.000000
50%      112000.000000
75%      161000.000000
max      200000.000000
Name: price, dtype: float64

In [10]:
category_price = (
    products
    .groupby('category', as_index=False)
    .agg(
        product_count=('product_id', 'count'),
        avg_price=('price', 'mean'),
        min_price=('price', 'min'),
        max_price=('price', 'max'),
    )
    .assign(avg_price=lambda df: df['avg_price'].round(0))
    .sort_values('avg_price', ascending=False)
)

category_price

,category,product_count,avg_price,min_price,max_price
4,식품,7,137143.0,67000,197000
1,뷰티,16,117688.0,5000,200000
6,패션,11,115909.0,28000,198000
3,스포츠,19,111579.0,10000,196000
0,도서,14,106857.0,25000,174000
5,전자기기,17,101588.0,5000,165000
2,생활용품,16,96438.0,11000,188000


### 결과 관찰

상품은 스포츠가 19개(19%)로 제일 많고 식품이 7개(7%)로 제일 적었다. 가격은 5천 원부터 20만 원까지 있고 평균은 11만 원 정도였다. 그런데 카테고리별 평균 가격을 보니 식품이 137,143원으로 오히려 제일 비싸고, 생활용품이 96,438원으로 제일 쌌다.

상품 수가 제일 적은 카테고리가 평균 단가는 제일 높다는 게 좀 의외였다. 상품 수만 보고 그 카테고리 매출이 작을 거라고 미리 판단하면 안 될 것 같다는 생각이 들었다. 이 조합이 실제 매출에서 어떻게 나타나는지는 뒤에서 category_sales를 보고 같이 확인해야 한다(-> 8번, 과제 2).

## 7. 주문 데이터 EDA

In [11]:
order_status = orders['order_status'].value_counts(dropna=False).reset_index()
order_status.columns = ['order_status', 'order_count']
order_status['order_ratio'] = (
    order_status['order_count'] / order_status['order_count'].sum() * 100
).round(2)

order_status

,order_status,order_count,order_ratio
0,completed,184,61.33
1,cancelled,64,21.33
2,refunded,52,17.33


In [12]:
payment_method = orders['payment_method'].value_counts(dropna=False).reset_index()
payment_method.columns = ['payment_method', 'order_count']
payment_method['order_ratio'] = (
    payment_method['order_count'] / payment_method['order_count'].sum() * 100
).round(2)

payment_method

,payment_method,order_count,order_ratio
0,kakao_pay,79,26.33
1,naver_pay,77,25.67
2,bank_transfer,74,24.67
3,card,70,23.33


In [13]:
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())
print('주문 월 개수:', orders['order_month'].nunique())

주문 시작일: 2025-07-09 00:00:00
주문 종료일: 2026-07-08 00:00:00
주문 월 개수: 13


### 결과 관찰

주문 300건 중 completed가 184건(61.33%), cancelled가 64건(21.33%), refunded가 52건(17.33%)이었다. 결제수단은 kakao_pay, naver_pay, bank_transfer, card 네 가지가 23~26% 사이에서 비슷하게 나왔다. 주문 기간은 2025-07부터 2026-07까지 13개월이었다.

completed가 61.33%밖에 안 된다는 게 눈에 들어왔다. 취소나 환불까지 합치면 전체의 38.67%나 되는데, 이걸 빼놓지 않고 전체 금액을 그냥 매출이라고 부르면 실제보다 꽤 크게 잡힐 것 같았다. 그래서 다음 섹션에서 completed로 필터링한 금액과 전체 금액을 직접 비교해보기로 했다.

## 8. 주문 상세 금액 계산과 completed 필터링

`line_total`을 만들고 `order_items`와 `orders`를 `order_id` 기준으로 합친 다음, 매출 관련 분석은 전부 `order_status == 'completed'` 범위로 좁혔다.

In [14]:
if 'line_total' not in order_items.columns:
    order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum())

order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_month', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())
order_sales = order_sales.drop(columns='_merge')

orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

In [15]:
all_status_total = order_sales['line_total'].sum()

completed_order_sales = order_sales[order_sales['order_status'] == 'completed'].copy()
completed_total = completed_order_sales['line_total'].sum()

print('전체 주문 상세 행 수:', len(order_sales))
print('completed 주문 상세 행 수:', len(completed_order_sales))
print('전체 상태 포함 line_total 합계:', f'{all_status_total:,.0f}')
print('completed line_total 합계:     ', f'{completed_total:,.0f}')
print('취소/환불 제외 금액:            ', f'{all_status_total - completed_total:,.0f}')

전체 주문 상세 행 수: 764
completed 주문 상세 행 수: 474
전체 상태 포함 line_total 합계: 255,610,000
completed line_total 합계:      148,990,000
취소/환불 제외 금액:             106,620,000


### 결과 관찰

order_items와 orders를 order_id로 합쳤는데 764행에서 764행으로 그대로였고 `_merge`도 전부 `both`였다. 전체 상태를 다 포함한 line_total 합계는 255,610,000원이었는데, completed만 남기니 148,990,000원(474행)으로 줄었다. 차이가 106,620,000원, 전체의 41.7% 정도였다.

7번 섹션에서 걱정했던 게 실제로 맞았다. 전체 금액으로 매출을 잡으면 completed 기준보다 1.72배(255.6백만 / 149.0백만) 정도 크게 나온다. 그래서 이 뒤로 나오는 카테고리별·월별·고객별 매출은 다 이 148,990,000원을 기준으로 삼고, 나눠서 더해도 이 숫자가 그대로 나오는지 끝에서 확인하기로 했다(-> 11번 섹션).

## 9. 카테고리별 completed 매출

`completed_order_sales`에 상품 정보를 연결해서 카테고리별 매출을 계산했다.

In [16]:
print('products.product_id 중복 수:', products['product_id'].duplicated().sum())

completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts())
completed_sales_items = completed_sales_items.drop(columns='_merge')

products.product_id 중복 수: 0
병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

In [17]:
category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

category_sales_with_price = category_sales.merge(
    category_price[['category', 'avg_price']],
    on='category',
    how='left',
)

category_sales_with_price

,category,total_quantity,total_sales,sales_ratio,avg_price
0,스포츠,295,31743000,21.31,111579.0
1,전자기기,259,26400000,17.72,101588.0
2,생활용품,272,23915000,16.05,96438.0
3,뷰티,223,23383000,15.69,117688.0
4,식품,133,16573000,11.12,137143.0
5,도서,149,16389000,11.00,106857.0
6,패션,111,10587000,7.11,115909.0


### 결과 관찰

카테고리별 completed 매출 비중은 스포츠 21.31%, 전자기기 17.72%, 생활용품 16.05%, 뷰티 15.69%, 식품 11.12%, 도서 11.00%, 패션 7.11% 순이었다. 스포츠는 판매 수량(295개)도 제일 많았다. 그런데 패션은 평균 단가가 115,909원으로 7개 카테고리 중 3번째로 높은데 매출 비중은 제일 낮았다.

6번 섹션에서 상품 수와 평균 단가가 꼭 같이 가지 않는다는 걸 봤는데, 여기서도 비슷한 게 보인다. 패션이 단가는 꽤 높은 편인데 매출이 낮은 이유가 뭘까 싶어서 `total_quantity`를 다시 보니 111개로 7개 카테고리 중 가장 적었다. 그러니까 단가가 낮아서 매출이 작은 게 아니라 판매량 자체가 적어서 그런 걸로 보인다. 다만 왜 판매량이 적은지 — 선호도 때문인지, 재고가 부족했던 건지, 노출이 안 됐던 건지 — 는 이 데이터만 봐서는 알 수가 없다.

## 10. 월별 completed 매출

In [18]:
monthly_sales = (
    completed_order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_sales['avg_order_value'] = (
    monthly_sales['total_sales'] / monthly_sales['order_count']
).round(0)

monthly_sales

,order_month,total_sales,order_count,avg_order_value
0,2025-07,5869000,8,733625.0
1,2025-08,15621000,18,867833.0
2,2025-09,10190000,13,783846.0
3,2025-10,25766000,26,991000.0
4,2025-11,8812000,12,734333.0
5,2025-12,11501000,14,821500.0
6,2026-01,17423000,22,791955.0
7,2026-02,9749000,17,573471.0
8,2026-03,13429000,14,959214.0
9,2026-04,17553000,23,763174.0


### 결과 관찰

월별 completed 매출은 2025-10이 25,766,000원(26건)으로 제일 높고 2026-07이 2,188,000원(2건)으로 제일 낮았다. 평균 주문 금액은 월마다 573,471원에서 1,094,000원 사이를 오갔다.

2026-06, 2026-07은 다른 달보다 주문 건수 자체가 몇 건밖에 안 돼서, 이 두 달의 평균 주문 금액이 높게 나온 것도 그냥 표본이 적어서 생긴 걸로 보는 게 맞을 것 같다. 최근 들어 매출이 줄었다거나 건당 금액이 늘었다고 바로 말하기보다는, 데이터 수집 기간이 그 근처에서 끝난 건 아닌지부터 먼저 봐야 할 것 같다.

## 11. 고객별 completed 구매 금액

출력에는 실제 이름 대신 익명화된 라벨을 썼다.

In [19]:
customer_sales = (
    completed_order_sales
    .groupby('customer_id', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales = customer_sales.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='one_to_one',
)

customer_sales['avg_order_value'] = (
    customer_sales['total_sales'] / customer_sales['order_count']
).round(0)
customer_sales['customer_label'] = 'Customer ' + customer_sales['customer_id'].astype(str)

customer_sales[['customer_label', 'city', 'order_count', 'total_sales', 'avg_order_value']].head(10)

,customer_label,city,order_count,total_sales,avg_order_value
0,Customer 117,성남,5,4100000,820000.0
1,Customer 102,고양,4,3996000,999000.0
2,Customer 83,수원,4,3880000,970000.0
3,Customer 30,서울,5,3590000,718000.0
4,Customer 40,서울,4,3523000,880750.0
5,Customer 20,인천,2,3191000,1595500.0
6,Customer 3,성남,2,3178000,1589000.0
7,Customer 111,광주,3,3153000,1051000.0
8,Customer 66,서울,4,3093000,773250.0
9,Customer 147,부산,2,2990000,1495000.0


### 결과 관찰

completed 구매 금액 상위 10명을 보면 Customer 20, Customer 3, Customer 147처럼 주문이 2건뿐인데 건당 평균이 150만 원대인 고객이 있고, Customer 117처럼 주문 5건에 건당 평균 82만 원인 고객도 있다. `total_sales`로만 줄을 세우면 이 둘이 비슷한 등수에 들어온다.

총 구매 금액만 보면 둘 다 우수 고객처럼 보이는데 성격이 완전히 다르다는 생각이 들었다. 한 번에 크게 산 고객과 여러 번 나눠서 산 고객을 같은 기준으로 묶는 게 맞는지 의문이 들어서, `order_count`와 `avg_order_value`를 같이 봐야 한다고 판단했다(-> 과제 5에서 기준을 정리했다).

## 12. 집계 총합 검증

카테고리별, 월별, 고객별 합계가 모두 `completed_order_sales`의 원본 `line_total` 합계와 일치하는지 확인했다.

In [20]:
base = completed_order_sales['line_total'].sum()
cat_sum = category_sales['total_sales'].sum()
monthly_sum = monthly_sales['total_sales'].sum()
cust_sum = customer_sales['total_sales'].sum()

verify = pd.DataFrame({
    'check': ['카테고리별 합계', '월별 합계', '고객별 합계'],
    'value': [cat_sum, monthly_sum, cust_sum],
    'match_base': [cat_sum == base, monthly_sum == base, cust_sum == base],
})

print('completed 원본 line_total 합계:', f'{base:,.0f}')
verify

completed 원본 line_total 합계: 148,990,000


,check,value,match_base
0,카테고리별 합계,148990000,True
1,월별 합계,148990000,True
2,고객별 합계,148990000,True


### 결과 관찰

completed 원본 line_total 합계는 148,990,000원이었고, 카테고리별·월별·고객별로 나눠서 더한 합계도 전부 148,990,000원으로 똑같이 나왔다(`match_base` 전부 True).

세 집계 다 같은 `completed_order_sales`에서 그룹만 바꿔서 더한 거라 당연히 같아야 하는데, 실제로 숫자가 맞아떨어지는 걸 보고 나니 중간에 중복으로 합쳐지거나 빠진 행이 없었다는 게 확인된 셈이다. 만약 하나라도 안 맞았으면 merge 단계에서 중복 매칭이 됐는지부터 다시 봤을 것 같다.

## 13. EDA 결과를 다음 질문으로 연결하기

In [21]:
eda_result_summary = pd.DataFrame({
    'question': [
        '고객은 어느 도시에 많이 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '카테고리별 completed 매출은 어떻게 다른가?',
        '월별 completed 매출과 주문 수는 어떻게 변하는가?',
        'completed 구매 금액이 높은 고객은 누구인가?',
    ],
    'result_table': [
        'customer_city',
        'product_category',
        'category_sales',
        'monthly_sales',
        'customer_sales',
    ],
    'next_question': [
        '도시별 completed 구매 금액도 차이가 있는가?',
        '상품 수가 많은 카테고리가 매출도 높은가?',
        '매출 차이가 수량 때문인가 단가 때문인가?',
        '특정 월의 매출 변화는 어떤 카테고리 때문인가?',
        '고액 구매 고객은 반복 구매 고객인가, 단발 고액 구매 고객인가?',
    ],
})

eda_result_summary

,question,result_table,next_question
0,고객은 어느 도시에 많이 분포하는가?,customer_city,도시별 completed 구매 금액도 차이가 있는가?
1,어떤 카테고리의 상품이 많은가?,product_category,상품 수가 많은 카테고리가 매출도 높은가?
2,카테고리별 completed 매출은 어떻게 다른가?,category_sales,매출 차이가 수량 때문인가 단가 때문인가?
3,월별 completed 매출과 주문 수는 어떻게 변하는가?,monthly_sales,특정 월의 매출 변화는 어떤 카테고리 때문인가?
4,completed 구매 금액이 높은 고객은 누구인가?,customer_sales,"고액 구매 고객은 반복 구매 고객인가, 단발 고액 구매 고객인가?"


## 14. EDA 결과와 보고서 저장하기

In [22]:
customer_city.to_csv(REPORT_DIR / 'ch06_customer_city.csv', index=False, encoding='utf-8-sig')
product_category.to_csv(REPORT_DIR / 'ch06_product_category.csv', index=False, encoding='utf-8-sig')
category_sales_with_price.to_csv(REPORT_DIR / 'ch06_category_sales.csv', index=False, encoding='utf-8-sig')
monthly_sales.to_csv(REPORT_DIR / 'ch06_monthly_sales.csv', index=False, encoding='utf-8-sig')
customer_sales.to_csv(REPORT_DIR / 'ch06_customer_sales.csv', index=False, encoding='utf-8-sig')
eda_result_summary.to_csv(REPORT_DIR / 'ch06_eda_questions.csv', index=False, encoding='utf-8-sig')

sorted(p.name for p in REPORT_DIR.glob('ch06_*.csv'))

['ch06_category_sales.csv',
 'ch06_customer_city.csv',
 'ch06_customer_sales.csv',
 'ch06_eda_questions.csv',
 'ch06_monthly_sales.csv',
 'ch06_product_category.csv']

In [23]:
summary_text = f"""# Chapter 6 EDA 요약 보고서

## 1. 분석 목적

전처리된 온라인 쇼핑몰 데이터를 사용해 고객, 상품, 주문, completed 매출 관점의 기본 현황을 탐색했습니다.

## 2. 금액성 분석 범위

order_status == 'completed' 기준. 전체 상태 포함 금액 {all_status_total:,.0f}원 중 completed {completed_total:,.0f}원.

## 3. 집계 총합 검증

```text
{verify.to_string(index=False)}
```

## 4. 카테고리별 completed 매출 요약

```text
{category_sales_with_price.to_string(index=False)}
```

## 5. 월별 completed 매출 요약

```text
{monthly_sales.to_string(index=False)}
```

## 6. 고객별 completed 구매 금액 상위 10명

```text
{customer_sales[['customer_label', 'city', 'order_count', 'total_sales', 'avg_order_value']].head(10).to_string(index=False)}
```

## 7. 추가 분석 질문

```text
{eda_result_summary.to_string(index=False)}
```

## 8. 해석 시 주의사항

- EDA 결과는 최종 결론이 아니라 추가 분석을 위한 관찰 결과입니다.
- 매출이 높은 카테고리가 반드시 선호도가 높은 카테고리라는 뜻은 아닙니다.
- completed 기준 금액은 할인, 배송비가 반영되지 않아 회계상 순매출과 다릅니다.
- 월별 매출 변화의 원인을 설명하려면 프로모션, 계절성 등 데이터에 없는 정보가 필요합니다.
"""

report_path = REPORT_DIR / 'ch06_eda_summary.md'
report_path.write_text(summary_text, encoding='utf-8')
print('EDA 요약 보고서 저장 완료:', report_path)

EDA 요약 보고서 저장 완료: C:\Users\uncle\Desktop\서울대 응용공학과\수업\2학기\대규모언어모델\git\llm-data-analysis-study\reports\ch06_eda_summary.md


### 결과 관찰

reports 폴더에 customer_city, product_category, category_sales, monthly_sales, customer_sales, eda_questions CSV와 ch06_eda_summary.md까지 전부 생성된 걸 확인했다.

이렇게 표를 CSV로 남겨두면 다음 장에서 그래프를 그릴 때 이번 장 로직을 또 돌릴 필요 없이 바로 가져다 쓸 수 있을 것 같다.

## 15. LLM으로 다음 질문 확장하기

### 사용 목적

여기까지 보고 나니 다음으로 뭘 더 봐야 할지가 하나로 딱 좁혀지지 않았다. 정답을 받으려는 게 아니라 내가 놓친 각도의 질문을 더 받아보려는 목적으로 LLM을 써봤다. 원본 데이터 대신 테이블 구조와 completed 기준 집계 요약만 보여줬다.

### Safe Context로 제공한 요약

```text
온라인 쇼핑몰 EDA 데이터 구조
- customers: customer_id, gender, age, city, signup_date (150행)
- products: product_id, product_name, category, price (100행)
- orders: order_id, customer_id, order_date, payment_method, order_status (300행)
- order_items: order_id, product_id, quantity, unit_price, line_total (764행)

completed 기준 집계 요약
- 카테고리별 completed 매출 비중: 스포츠 21.31% ~ 패션 7.11%
- 월별 completed 매출: 2025-10에 집중, 최근 1~2개월은 주문 건수가 적음
- 고객별 completed 구매 금액: 주문 2건에 건당 150만 원대인 고객과 주문 5건에 건당 82만 원대인 고객이 비슷한 총액으로 묶임
```

### 사용한 Prompt

```text
위 구조와 집계 요약만 보고 다음 분석 질문 후보 5개를 제안해줘.

조건:
- 지금 가진 데이터(customers, products, orders, order_items)로 실제로 계산 가능한 질문만 제안할 것
- 데이터에 없는 정보(프로모션, 광고, 재구매 의도 등)를 가정한 질문은 제안하지 말 것
- 각 질문에 필요한 컬럼과 pandas 연산을 같이 적어줄 것
```

### LLM 답변 요약

1. 도시별 completed 매출과 도시별 고객 수를 비교하면 어떤 차이가 있는가? (`customers.city` + `completed_order_sales` 병합)
2. 결제수단별 completed 평균 주문 금액은 어떻게 다른가? (`orders.payment_method` + `line_total`)
3. 고객의 가입일(`signup_date`)과 첫 구매일 간격이 구매 금액에 영향을 주는가? (`customers.signup_date`, `orders.order_date`)
4. 반복 구매 고객이 광고를 더 많이 본 고객인가?
5. 카테고리별 재구매율은 어떻게 다른가?

### 결과 관찰

5개 중 앞의 두 개는 지금 가진 컬럼으로 바로 계산할 수 있는 질문이었다. 그런데 3번은 "영향을 주는가"라는 표현이 들어 있었고, 4번은 광고 노출 데이터, 5번은 재구매 정의가 지금 테이블에 없는데도 그냥 제안에 들어가 있었다.

### 나의 해석과 판단

1번, 2번은 그대로 사용하기로 했다. 3번은 인과관계처럼 들리는 "영향을 주는가"를 "함께 변하는 경향이 있는가"로 바꿔서 상관관계 확인 수준으로 낮춰 쓰기로 했다. 4번, 5번은 광고 노출이나 재구매 정의처럼 지금 스키마에 없는 개념을 전제로 하고 있어서 보류하기로 했다. 조건에 "데이터에 없는 정보를 가정하지 말라"고 분명히 적었는데도 이런 제안이 나온 걸 보면, 조건을 적어도 LLM이 매번 완전히 지키는 건 아닌 것 같다. 결국 제안이 나온 뒤에 내가 컬럼이 실제로 있는지 하나씩 확인하는 과정은 생략할 수 없었다.

## 16. 실습 과제

아래 6가지 과제를 직접 해결한다.

### 과제 1. 도시별 고객 수와 도시별 completed 총 구매 금액을 비교하세요.

In [24]:
# 과제 1. 도시별 고객 수와 도시별 completed 총 구매 금액 비교
city_customer_sales = customer_sales.groupby('city', as_index=False).agg(
    total_sales=('total_sales', 'sum'),
    buyer_count=('customer_id', 'nunique'),
)

city_compare = customer_city.merge(city_customer_sales, on='city', how='left')
city_compare['total_sales'] = city_compare['total_sales'].fillna(0)
city_compare['buyer_count'] = city_compare['buyer_count'].fillna(0).astype(int)
city_compare['avg_sales_per_customer'] = (
    city_compare['total_sales'] / city_compare['customer_count']
).round(0)
city_compare = city_compare.sort_values('total_sales', ascending=False)

city_compare

,city,customer_count,customer_ratio,total_sales,buyer_count,avg_sales_per_customer
0,성남,21,14.00,24099000,17,1147571.0
9,고양,11,7.33,19116000,9,1737818.0
4,서울,15,10.00,19097000,12,1273133.0
5,울산,14,9.33,15380000,10,1098571.0
1,광주,17,11.33,14264000,11,839059.0
2,부산,16,10.67,13604000,8,850250.0
6,인천,14,9.33,12832000,10,916571.0
7,대전,14,9.33,11792000,8,842286.0
8,수원,13,8.67,10620000,7,816923.0
3,대구,15,10.00,8186000,8,545733.0


### 결과 관찰

고객 수 1위는 성남(21명)인데, 고객 1인당 평균 구매 금액은 고양(11명, 1,737,818원)이 제일 높았다. 대구는 고객 15명으로 중간 규모인데 1인당 평균 구매 금액(545,733원)은 전체 도시 중 가장 낮았다.

5번 섹션에서 예상했던 대로 고객 수와 구매 금액이 비례하지 않는다는 게 확인됐다. 고객 수만 보고 지역 전략을 세우면 고양처럼 고객은 적지만 구매력이 높은 지역을 놓칠 수 있을 것 같다.

### 과제 2. 상품 수가 많은 카테고리가 completed 매출도 높은지 확인하세요.

In [25]:
# 과제 2. 상품 수가 많은 카테고리가 completed 매출도 높은지 확인
category_compare = product_category.merge(
    category_sales[['category', 'total_sales', 'sales_ratio']],
    on='category',
    how='left',
).sort_values('product_count', ascending=False)

category_compare['rank_by_product_count'] = category_compare['product_count'].rank(ascending=False).astype(int)
category_compare['rank_by_sales'] = category_compare['total_sales'].rank(ascending=False).astype(int)

category_compare

,category,product_count,product_ratio,total_sales,sales_ratio,rank_by_product_count,rank_by_sales
0,스포츠,19,19.0,31743000,21.31,1,1
1,전자기기,17,17.0,26400000,17.72,2,2
2,생활용품,16,16.0,23915000,16.05,3,3
3,뷰티,16,16.0,23383000,15.69,3,4
4,도서,14,14.0,16389000,11.00,5,6
5,패션,11,11.0,10587000,7.11,6,7
6,식품,7,7.0,16573000,11.12,7,5


### 결과 관찰

상품 수 순위와 매출 순위는 스포츠(1위, 1위), 전자기기(2위, 2위)는 그대로 맞아떨어졌다. 그런데 식품은 상품 수 순위가 7위(제일 적음)인데 매출 순위는 5위로 두 단계 올라갔고, 패션은 상품 수 순위 6위인데 매출 순위는 7위(제일 낮음)로 내려갔다.

상품 수가 많으면 대체로 매출도 높은 편이긴 한데, 식품처럼 상품 수가 적어도 평균 단가가 높으면 순위가 뒤집힐 수 있다는 걸 다시 확인했다. "상품 수가 많다 = 매출이 높다"라고 단정하면 안 될 것 같다.

### 과제 3. 결제수단별 completed 평균 주문 금액을 계산하세요.

In [26]:
# 과제 3. 결제수단별 completed 평균 주문 금액
completed_order_sales_pm = completed_order_sales.merge(
    orders[['order_id', 'payment_method']],
    on='order_id',
    how='left',
)

payment_order_value = (
    completed_order_sales_pm
    .groupby(['payment_method', 'order_id'], as_index=False)
    .agg(order_amount=('line_total', 'sum'))
    .groupby('payment_method', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        avg_order_value=('order_amount', 'mean'),
    )
)
payment_order_value['avg_order_value'] = payment_order_value['avg_order_value'].round(0)
payment_order_value = payment_order_value.sort_values('avg_order_value', ascending=False)

payment_order_value

,payment_method,order_count,avg_order_value
3,naver_pay,51,852941.0
1,card,39,815538.0
2,kakao_pay,49,802898.0
0,bank_transfer,45,763156.0


### 결과 관찰

결제수단별 completed 평균 주문 금액은 naver_pay(852,941원), card(815,538원), kakao_pay(802,898원), bank_transfer(763,156원) 순이었다. 최고와 최저 차이가 9만 원 정도라 크게 벌어진 건 아니다.

결제수단에 따라 평균 주문 금액이 뚜렷하게 갈리지 않는 걸 보면, 결제수단이 구매 금액대를 설명하는 주요 요인은 아닌 것 같다.

### 과제 4. 월별·카테고리별 completed 매출 요약표를 만들어 보세요.

In [27]:
# 과제 4. 월별·카테고리별 completed 매출 요약표
monthly_category_sales = (
    completed_sales_items
    .groupby(['order_month', 'category'], as_index=False)
    .agg(total_sales=('line_total', 'sum'))
)

monthly_category_pivot = monthly_category_sales.pivot_table(
    index='order_month',
    columns='category',
    values='total_sales',
    fill_value=0,
).round(0)

monthly_category_pivot

category,도서,뷰티,생활용품,스포츠,식품,전자기기,패션
order_month,,,,,,,
2025-07,656000.0,1336000.0,1117000.0,432000.0,0.0,2328000.0,0.0
2025-08,528000.0,2561000.0,1513000.0,3613000.0,2635000.0,2760000.0,2011000.0
2025-09,844000.0,1573000.0,1250000.0,2815000.0,268000.0,2696000.0,744000.0
2025-10,2550000.0,4606000.0,6027000.0,5799000.0,1899000.0,4118000.0,767000.0
2025-11,870000.0,1317000.0,867000.0,1611000.0,1271000.0,2272000.0,604000.0
2025-12,1975000.0,2305000.0,1060000.0,1103000.0,2130000.0,2928000.0,0.0
2026-01,2352000.0,1497000.0,2138000.0,5966000.0,2083000.0,2192000.0,1195000.0
2026-02,1044000.0,1794000.0,918000.0,2656000.0,1597000.0,464000.0,1276000.0
2026-03,990000.0,1385000.0,2100000.0,3768000.0,2237000.0,1005000.0,1944000.0


### 결과 관찰

월별·카테고리별 표에서 2025-10의 생활용품(6,027,000원)이 전체 조합 중 가장 큰 값이었다. 2025-07의 식품(0원), 2026-06의 도서·식품(0원)처럼 0원인 칸도 여러 개 있었다.

2025-10은 생활용품뿐 아니라 여러 카테고리에서 같이 높게 나온 걸 보면, 특정 카테고리만의 효과가 아니라 그 달 전체 주문이 몰린 쪽에 가까운 것 같다(-> 10번 섹션 관찰과 비슷하다). 0원인 칸은 그 달에 그 카테고리의 completed 주문이 아예 없었다는 뜻이고 결측치는 아니다.

### 과제 5. 고객별 `order_count`와 `avg_order_value`를 함께 보고 반복 구매 고객과 고액 단발 구매 고객을 구분하는 기준을 생각해 보세요.

**답변:**

11번 섹션에서 봤듯 `total_sales`만으로는 적게 자주 사는 고객과 크게 가끔 사는 고객을 구분할 수 없었다. 그래서 기준을 이렇게 나눠봤다.

- 반복 구매 고객: `order_count`가 상위권(4건 이상)이면서 `avg_order_value`는 중간 수준인 고객. 예를 들어 Customer 117(5건, 건당 820,000원), Customer 102(4건, 건당 999,000원).
- 고액 단발 구매 고객: `order_count`가 1~2건으로 적은데 `avg_order_value`가 completed 평균 주문 금액(148,990,000원 / 184건 ≈ 809,728원)보다 훨씬 높은 고객. 예를 들어 Customer 20(2건, 건당 1,595,500원), Customer 147(2건, 건당 1,495,000원).

이 두 유형은 관리 방법이 같을 것 같지 않다. 반복 구매 고객한테는 적립이나 구독 같은 재구매 유도가 통할 수 있어 보이는데, 고액 단발 구매 고객의 큰 구매가 선물이나 이벤트성이라면 같은 전략이 똑같이 먹힐지는 지금 데이터만으로는 판단이 안 된다.

### 과제 6. LLM에게 현재 데이터로 답할 수 없는 질문을 답할 수 있는 질문으로 바꾸게 하는 프롬프트를 작성해 보세요.

**작성한 프롬프트:**

```text
다음은 온라인 쇼핑몰 completed 주문 기준 EDA 중 나온 질문이다.

원래 질문: "인기 상품은 왜 잘 팔릴까?"

이 질문은 지금 가진 데이터(customers, products, orders, order_items)만으로는
"왜"라는 원인을 설명할 수 없다.

요청:
1. 위 질문을 지금 데이터로 실제 계산 가능한 질문으로 바꿔줘.
2. 바꾼 질문에 필요한 컬럼과 pandas 연산(groupby, merge 등)을 같이 제시해줘.
3. 데이터에 없는 요인(브랜드 선호도, 리뷰, 광고 노출 등)은 가정하지 말아줘.
4. "원인을 설명한다"는 표현 대신 "경향을 확인한다"는 표현을 써줘.
```

**결과 관찰**

원래 질문은 "왜"를 묻고 있어서 지금 스키마로는 답이 안 나온다. 프롬프트에 데이터에 없는 요인을 가정하지 말라는 조건과 원인 대신 경향 확인이라는 조건을 넣었더니, `quantity`, `line_total`, `category` 같은 실제 컬럼 기준으로 측정 가능한 질문(판매량 상위 상품은 평균 단가가 낮은 편인가?)으로 바뀌어서 나왔다.

## Chapter 06 최종 인사이트

### 가장 의미 있다고 생각한 결과 2~3가지

첫 번째는 completed 필터링 전후 매출 차이다. 전체 상태를 다 포함하면 255,610,000원인데 completed만 보면 148,990,000원으로, 거의 1.72배 차이가 난다. "매출"을 어떤 기준으로 집계하느냐에 따라 숫자가 이렇게 달라질 수 있다는 걸 직접 숫자로 확인한 게 이번 장에서 제일 크게 느낀 부분이다.

두 번째는 패션 카테고리다. 평균 단가는 115,909원으로 7개 중 3위인데 매출 비중은 7.11%로 제일 낮다. 단가가 아니라 판매 수량(111개, 최저)이 부족해서 그런 걸로 보이는데, 처음엔 단가가 높으면 매출도 높을 거라고 막연히 생각했다가 수량을 직접 보고 나서야 그 생각이 틀렸다는 걸 알았다.

세 번째는 구매 금액 상위 고객 안에 성격이 다른 두 그룹이 섞여 있었다는 점이다. 주문 2건에 건당 150만 원대인 고객과 주문 5건에 건당 82만 원대인 고객이 `total_sales` 기준으로는 비슷한 줄에 서 있었다.

### 각 관찰을 설명할 수 있는 가설

취소·환불 비율(38.67%)이 높은 이유는 이 샘플 데이터가 만들어진 방식 자체일 수도 있고, 실제 서비스라면 특정 카테고리나 결제수단에서 취소율이 더 높을 수도 있다. 패션 판매 수량이 적은 이유는 선호도, 재고, 노출 중 뭐가 맞는지 모르겠다. 고액 단발 구매는 선물이나 이벤트성 구매, 반복 구매는 생활 소모품성 구매일 가능성이 있어 보이는데 이건 그냥 추측에 가깝다.

### 현재 데이터만으로 단정할 수 없는 것

취소·환불이 왜 일어났는지, 패션 판매 수량이 왜 적은지, 2025-10 매출이 왜 유독 높았는지, 고액 단발 구매 고객이 나중에 다시 구매할지는 지금 가진 4개 테이블만으로는 알 수 없다.

### 다음 분석 우선순위

1. 카테고리별·결제수단별 취소·환불 비율을 나눠서 봐야 할 것 같다. 전체 취소율 38.67%가 어디서 제일 많이 나오는지부터 보고 싶다.
2. 2025-10 매출이 왜 유독 높았는지 — 신규 고객이 늘어서인지 기존 고객이 더 많이 샀는지 — 를 나눠서 봐야 한다.
3. 고액 단발 구매 고객이 시간이 지나면 다시 구매하는지를 추적해보고 싶은데, 지금은 데이터 기간이 짧아서 당장은 어렵다.

### 업무적으로 확인할 가치가 있는 질문

패션 카테고리 판매 수량이 왜 낮은지(재고, 마케팅 노출, 가격 경쟁력 중 뭔지)는 담당 부서에 직접 물어보는 게 데이터만으로 추측하는 것보다 나을 것 같다. 고액 단발 구매 고객한테 재구매를 유도하는 캠페인이 실제로 효과가 있는지도 A/B 테스트로 한번 확인해볼 만하다고 생각한다.